
# End-to-End EEG Deep Learning Pipeline
*Generated:* 2025-08-23 17:04:14

This notebook demonstrates a **complete pipeline**:
1. **Loading** EEG (BDF/EDF) or using **demo** data
2. **Visualization & inspection** (basic plots, PSDs, event overview)
3. **Preprocessing** (filter, resample, montage, event detection, epoching, baseline)
4. **More visualization** (ERP/evoked, PSD per class)
5. **Deep Learning models**: **DNN**, **CNN**, **CNN–DNN Hybrid**
6. **Performance metrics**: accuracy, classification report, confusion matrix; learning curves

> Works even without a real EEG file by switching to a demo generator.


## 0) Parameters

In [ ]:
# === Parameters (edit these) ===
DATA_PATH = ""  # e.g., r"/path/to/file.bdf"  (leave empty to run demo)
EVENT_CHANNEL_CANDIDATES = ["Status", "STI 014", "STI101", "TRIG", "Trigger"]
HPF, LPF = 0.5, 40.0             # band-pass
RESAMPLE_HZ = 128                # resample frequency
TMIN, TMAX = -0.2, 0.8           # epoch window (s)
BASELINE = (None, 0)             # baseline from start to 0 s
MIN_EPOCHS_PER_CLASS = 8
RANDOM_STATE = 42
N_EPOCHS_DEMO = 120              # demo synthetic epochs (if no file)
CLASSES_DEMO = 2                 # demo class count


## 1) Imports (graceful fallbacks)

In [ ]:
import os, warnings, math
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

try:
    import mne
except Exception as e:
    print("Warning: mne not available; file reading/epoching disabled.", e)
    mne = None

# ML / Metrics
try:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
except Exception as e:
    print("Warning: scikit-learn not available; metrics limited.", e)
    train_test_split = classification_report = confusion_matrix = ConfusionMatrixDisplay = None

# Deep learning
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv1D, MaxPooling1D
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.utils import to_categorical
except Exception as e:
    print("Warning: TensorFlow/Keras not available; DL sections will be skipped.", e)
    Sequential = Dense = Dropout = Flatten = Conv1D = MaxPooling1D = Adam = to_categorical = None


## 2) Load EEG (or fall back to a realistic demo)

In [ ]:
raw = None
sfreq = None

if mne is not None and DATA_PATH and os.path.exists(DATA_PATH):
    ext = os.path.splitext(DATA_PATH)[1].lower()
    print(f"Loading: {DATA_PATH}")
    if ext == ".bdf":
        raw = mne.io.read_raw_bdf(DATA_PATH, preload=True, verbose=False)
    elif ext in [".edf", ".set", ".fif"]:
        if ext == ".edf":
            raw = mne.io.read_raw_edf(DATA_PATH, preload=True, verbose=False)
        elif ext == ".set":
            raw = mne.io.read_raw_eeglab(DATA_PATH, preload=True, verbose=False)
        else:
            raw = mne.io.read_raw_fif(DATA_PATH, preload=True, verbose=False)
    else:
        raise ValueError("Unsupported EEG file extension.")
    sfreq = raw.info['sfreq']
else:
    # DEMO: synthesize multi-channel EEG-like rhythmic data with two classes
    print("No valid EEG file provided. Using demo data.")
    sfreq = RESAMPLE_HZ
    dur_s = 60
    times = np.arange(0, dur_s, 1/sfreq)
    n_channels = 16
    # Build alternating alpha/beta bursts to mimic two classes
    sig_alpha = np.sin(2*np.pi*10*times) * (1 + 0.2*np.random.randn(times.size))
    sig_beta  = np.sin(2*np.pi*20*times) * (1 + 0.2*np.random.randn(times.size))
    data = np.vstack([sig_alpha + 0.1*np.random.randn(times.size) for _ in range(n_channels)])
    info = mne.create_info([f"EEG{i}" for i in range(n_channels)], sfreq, ch_types="eeg") if mne else None
    raw = mne.io.RawArray(data, info) if mne else None
    # add fake annotations for two classes alternating every ~1.0s after a start offset
    if mne:
        onsets = np.arange(1, dur_s-1, 1.0)
        desc = ["A" if i%2==0 else "B" for i in range(len(onsets))]
        annot = mne.Annotations(onset=onsets, duration=[0.1]*len(onsets), description=desc)
        raw.set_annotations(annot)


## 3) Inspect & visualize raw

In [ ]:
if raw is not None:
    print(raw)
    try:
        raw.set_montage("standard_1020", on_missing="ignore")
    except Exception as e:
        print("Montage set failed:", e)
    # Basic PSD plot
    fig = raw.compute_psd(fmin=1, fmax=45).plot(show=False)
    plt.show(fig)
    # Quick snapshot (first 5 channels) using matplotlib to avoid interactive backends
    n_plot = min(5, len(raw.ch_names))
    data, times = raw[:n_plot, :int(sfreq*5)]  # 5 seconds
    plt.figure(figsize=(10, 4))
    for i in range(n_plot):
        plt.plot(times, data[i] + i*5*np.std(data), label=raw.ch_names[i])
    plt.xlabel("Time (s)"); plt.ylabel("Amplitude (offset per channel)"); plt.title("Raw snapshot")
    plt.legend(loc="upper right"); plt.tight_layout(); plt.show()
else:
    print("No raw available for visualization.")


## 4) Preprocessing

In [ ]:
if raw is not None and mne is not None:
    if HPF or LPF:
        raw.filter(l_freq=HPF, h_freq=LPF, method="iir", verbose=False)
    if RESAMPLE_HZ:
        raw.resample(RESAMPLE_HZ, npad="auto", verbose=False)
    print("Preprocessing done:", raw)

    # Event detection: try stim channels, else annotations
    events, event_id = None, None
    for stim in EVENT_CHANNEL_CANDIDATES:
        try:
            ev = mne.find_events(raw, stim_channel=stim, shortest_event=1, verbose=False)
            if len(ev) > 0:
                events = ev
                print(f"Found {len(events)} events on channel '{stim}'.")
                break
        except Exception:
            pass
    if (events is None or len(events)==0) and raw.annotations is not None and len(raw.annotations)>0:
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        print(f"Events from annotations: {len(events)}")

    if events is not None and (event_id is None or len(event_id)==0):
        # Auto-label from codes
        uniq = np.unique(events[:,2])
        event_id = {f"L{int(k)}": int(k) for k in uniq}
        print("Auto event_id:", event_id)

    if events is None or len(events)==0:
        raise RuntimeError("No events found. Provide a file with events/annotations.")
else:
    raise RuntimeError("MNE/raw not available.")


## 5) Epoching

In [ ]:
epochs = None
if raw is not None and mne is not None:
    picks = mne.pick_types(raw.info, eeg=True, eog=False, stim=False, exclude='bads')
    epochs = mne.Epochs(raw, events=events, event_id=event_id, tmin=TMIN, tmax=TMAX,
                        baseline=BASELINE, picks=picks, preload=True, verbose=False)
    print(epochs)
    # Balance: check counts
    y_evt = epochs.events[:,2]
    counts = {label: int(np.sum(y_evt==code)) for label, code in event_id.items()}
    print("Counts per class:", counts)
    if len(counts) < 2:
        raise RuntimeError("Need at least 2 classes for classification.")


## 6) More visualizing: Evoked responses & PSD per class

In [ ]:
if epochs is not None:
    # Evoked/ERP per class
    for label, code in event_id.items():
        ev = epochs[label].average()
        fig = ev.plot(spatial_colors=True, show=False, time_unit="s")
        plt.show(fig)

    # PSD per class (mean over epochs for first channel)
    ch_idx = 0
    plt.figure(figsize=(8,4))
    for label, code in event_id.items():
        dat = epochs[label].get_data()[:, ch_idx, :]  # (n_ep, n_times)
        psd = np.abs(np.fft.rfft(dat, axis=1))**2
        freqs = np.fft.rfftfreq(dat.shape[1], d=1.0/epochs.info["sfreq"])
        plt.plot(freqs, psd.mean(axis=0), label=label)
    plt.xlim(0, 45); plt.xlabel("Hz"); plt.ylabel("Power"); plt.title("PSD per class (ch 0)")
    plt.legend(); plt.tight_layout(); plt.show()


## 7) Prepare data tensors for DL

In [ ]:
# X_raw: (n_epochs, n_channels, n_times); y_raw: integer labels starting at 0
X_raw = epochs.get_data()
y_raw = epochs.events[:, 2]

# Normalize per-epoch, per-channel
X_raw = (X_raw - X_raw.mean(axis=-1, keepdims=True)) / (X_raw.std(axis=-1, keepdims=True) + 1e-6)

# Reindex labels to start at 0 and one-hot encode
label_map = {code:i for i, code in enumerate(sorted(np.unique(y_raw)))}
y_int = np.array([label_map[v] for v in y_raw])
n_classes = len(np.unique(y_int))

if to_categorical is not None:
    y_onehot = to_categorical(y_int, n_classes)
else:
    y_onehot = None

print("Data shape:", X_raw.shape, "Classes:", n_classes, label_map)


## 8) Train/Test Split

In [ ]:
if train_test_split is None:
    raise RuntimeError("scikit-learn is required for splitting and metrics.")

X_train, X_test, y_train, y_test, yi_train, yi_test = train_test_split(
    X_raw, y_onehot, y_int, test_size=0.2, random_state=RANDOM_STATE, stratify=y_int
)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 9) DNN on flattened epochs (with metrics & curves)

In [ ]:
if Sequential is None:
    raise RuntimeError("TensorFlow/Keras required for DL models.")

# Flatten
Xtr_flat = X_train.reshape(X_train.shape[0], -1)
Xte_flat = X_test.reshape(X_test.shape[0], -1)

dnn = Sequential([
    Dense(512, activation='relu', input_shape=(Xtr_flat.shape[1],)),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(n_classes, activation='softmax')
])
dnn.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
hist_dnn = dnn.fit(Xtr_flat, y_train, validation_data=(Xte_flat, y_test),
                   epochs=15, batch_size=32, verbose=0)

# Metrics
dnn_eval = dnn.evaluate(Xte_flat, y_test, verbose=0)
print(f"DNN — Loss: {dnn_eval[0]:.4f}, Acc: {dnn_eval[1]:.4f}")

# Learning curves
plt.figure()
plt.plot(hist_dnn.history['accuracy'], label='train_acc')
plt.plot(hist_dnn.history['val_accuracy'], label='val_acc')
plt.title("DNN Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Acc"); plt.legend(); plt.show()

# Confusion matrix
y_pred_dnn = dnn.predict(Xte_flat, verbose=0).argmax(axis=1)
if classification_report is not None:
    print(classification_report(yi_test, y_pred_dnn))
    cm = confusion_matrix(yi_test, y_pred_dnn)
    ConfusionMatrixDisplay(cm).plot()
    plt.title("DNN Confusion Matrix"); plt.show()


## 10) CNN on (time × channels) tensors (metrics & curves)

In [ ]:
# Reshape to (epochs, time, channels)
Xtr_cnn = np.transpose(X_train, (0, 2, 1))
Xte_cnn  = np.transpose(X_test, (0, 2, 1))

cnn = Sequential([
    Conv1D(32, kernel_size=7, activation='relu', input_shape=Xtr_cnn.shape[1:]),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(n_classes, activation='softmax')
])
cnn.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
hist_cnn = cnn.fit(Xtr_cnn, y_train, validation_data=(Xte_cnn, y_test),
                   epochs=15, batch_size=32, verbose=0)

cnn_eval = cnn.evaluate(Xte_cnn, y_test, verbose=0)
print(f"CNN — Loss: {cnn_eval[0]:.4f}, Acc: {cnn_eval[1]:.4f}")

plt.figure()
plt.plot(hist_cnn.history['accuracy'], label='train_acc')
plt.plot(hist_cnn.history['val_accuracy'], label='val_acc')
plt.title("CNN Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Acc"); plt.legend(); plt.show()

y_pred_cnn = cnn.predict(Xte_cnn, verbose=0).argmax(axis=1)
if classification_report is not None:
    print(classification_report(yi_test, y_pred_cnn))
    cm = confusion_matrix(yi_test, y_pred_cnn)
    ConfusionMatrixDisplay(cm).plot()
    plt.title("CNN Confusion Matrix"); plt.show()


## 11) CNN–DNN Hybrid (metrics & curves)

In [ ]:
hybrid = Sequential([
    Conv1D(32, kernel_size=7, activation='relu', input_shape=Xtr_cnn.shape[1:]),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(n_classes, activation='softmax')
])
hybrid.compile(optimizer=Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
hist_h = hybrid.fit(Xtr_cnn, y_train, validation_data=(Xte_cnn, y_test),
                    epochs=15, batch_size=32, verbose=0)

hyb_eval = hybrid.evaluate(Xte_cnn, y_test, verbose=0)
print(f"Hybrid — Loss: {hyb_eval[0]:.4f}, Acc: {hyb_eval[1]:.4f}")

plt.figure()
plt.plot(hist_h.history['accuracy'], label='train_acc')
plt.plot(hist_h.history['val_accuracy'], label='val_acc')
plt.title("Hybrid Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Acc"); plt.legend(); plt.show()

y_pred_h = hybrid.predict(Xte_cnn, verbose=0).argmax(axis=1)
if classification_report is not None:
    print(classification_report(yi_test, y_pred_h))
    cm = confusion_matrix(yi_test, y_pred_h)
    ConfusionMatrixDisplay(cm).plot()
    plt.title("Hybrid Confusion Matrix"); plt.show()


## 12) Save artifacts

In [ ]:
np.savez('/mnt/data/eeg_dl_artifacts.npz',
           X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
           yi_train=yi_train, yi_test=yi_test)
print("Saved tensors and labels to /mnt/data/eeg_dl_artifacts.npz")
